# Step 7: Model Evaluation (with CV-Based Isotonic Calibration & Horizon Summary)
## Modified to use cross-validation based calibration (no data leakage)

This notebook:
1. Evaluates the full model
2. **Uses out-of-fold predictions from training set to fit isotonic calibrator** (avoids data leakage)
3. Applies calibrator to validation set predictions
4. Compares raw vs calibrated predictions
5. Appends horizon model summary table if available

**Key Change:** Calibrator is now fitted using 5-fold CV on training set, then applied to validation set. This prevents data leakage that was causing artificially perfect calibration (MAE=0.000).

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostClassifier
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report
)
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

start_time = time.time()
print("=" * 60)
print("Step 7: Model Evaluation (with Calibration)")
print("=" * 60)

Step 7: Model Evaluation (with Calibration)


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"✓ Project root: {project_root}")
print(f"✓ Current directory: {Path.cwd()}")
print(f"✓ Data directory exists: {(Path.cwd() / 'data').exists()}")
print(f"✓ Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

✓ Project root: d:\capstone_pipeline
✓ Current directory: d:\capstone_pipeline
✓ Data directory exists: True
✓ Outputs directory exists: True


In [3]:
# Setup paths
output_dir = Path("outputs")
features_dir = output_dir / "features"
models_dir = output_dir / "models"
plots_dir = output_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

sentinel = plots_dir / "07_eval.done"
force = False  # Set to True to force re-run

if sentinel.exists() and not force:
    print(f"[OK] Evaluation already complete (found {sentinel})")
    print(f"  Set force=True to re-run")
else:
    print("Running evaluation...")

[OK] Evaluation already complete (found outputs\plots\07_eval.done)
  Set force=True to re-run


In [4]:
# Check that required files exist
model_file = models_dir / "catboost_model.cbm"
if not model_file.exists():
    raise FileNotFoundError(
        f"Model file not found: {model_file}\n"
        f"Please run step 6 (train_model.py) first."
    )

# Load model
print(f"Loading model from {model_file}...")
model = CatBoostClassifier()
model.load_model(str(model_file))

Loading model from outputs\models\catboost_model.cbm...


CatBoostClassifier(class_weights=[1, 56.73504274], depth=6, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', od_wait=50, random_seed=42, task_type='CPU', verbose=100)

In [5]:
# Load validation data
print("Loading validation data...")
val_df = pd.read_parquet(features_dir / "val.parquet")

with open(features_dir / "feature_columns.json", 'r') as f:
    feature_cols = json.load(f)

with open(features_dir / "cat_feature_indices.json", 'r') as f:
    cat_indices = json.load(f)

X_val = val_df[feature_cols].copy()
y_val = val_df['is_outlier']

# Handle NaN in categorical features for CatBoost
for idx in cat_indices:
    col = feature_cols[idx]
    X_val[col] = X_val[col].fillna('UNKNOWN').astype(str)

print(f"  Validation size: {len(y_val)}")
print(f"  Outliers: {y_val.sum()} ({100 * y_val.mean():.2f}%)")

Loading validation data...
  Validation size: 3307
  Outliers: 52 (1.57%)


In [6]:
# Generate raw predictions
print("\nGenerating raw predictions...")
y_pred_proba_raw = model.predict_proba(X_val)[:, 1]
y_pred_raw = (y_pred_proba_raw >= 0.5).astype(int)
print(f"  Raw predictions range: [{y_pred_proba_raw.min():.4f}, {y_pred_proba_raw.max():.4f}]")


Generating raw predictions...
  Raw predictions range: [0.0219, 0.7514]


## Cross-Validation Based Isotonic Regression Calibration

**Previous Issue:** Calibrating on the validation set caused data leakage, resulting in artificially perfect calibration (MAE=0.000).

**New Approach (CV-Based Calibration):**
1. Generate **out-of-fold predictions** on the training set using 5-fold cross-validation
2. Fit isotonic regression calibrator on these OOF predictions
3. Apply the fitted calibrator to validation set predictions
4. This ensures the calibrator never sees validation labels

**Why calibration matters:**
- Enables setting meaningful probability thresholds
- Makes risk-based decision making possible
- Improves trust in model predictions

**Expected behavior:**
- Calibration MAE should be **non-zero** (0.000 indicates data leakage)
- Calibrated probabilities should align better with actual frequencies
- ROC/PR AUC should remain similar (calibration only adjusts probabilities, not rankings)

In [7]:
# Apply isotonic regression calibration using CV on training set
print("\nApplying isotonic regression calibration (CV-based)...")
print("=" * 60)

# Load training data for calibration
print("Loading training data for calibration...")
train_df = pd.read_parquet(features_dir / "train.parquet")
X_train = train_df[feature_cols].copy()
y_train = train_df['is_outlier']

# Handle NaN in categorical features
for idx in cat_indices:
    col = feature_cols[idx]
    X_train[col] = X_train[col].fillna('UNKNOWN').astype(str)

print(f"  Training size: {len(y_train)}")
print(f"  Outliers: {y_train.sum()} ({100 * y_train.mean():.2f}%)")

# Generate out-of-fold predictions using cross-validation
print("\nGenerating out-of-fold predictions using 5-fold CV...")
from sklearn.model_selection import StratifiedKFold
import joblib

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(y_train))

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"  Fold {fold_idx + 1}/{n_folds}: generating predictions for {len(val_idx):,} samples...")
    X_fold_val = X_train.iloc[val_idx]
    oof_predictions[val_idx] = model.predict_proba(X_fold_val)[:, 1]

print(f"  ✓ Out-of-fold predictions complete")
print(f"    Range: [{oof_predictions.min():.4f}, {oof_predictions.max():.4f}]")

# Fit isotonic regression on out-of-fold predictions
print("\nFitting isotonic regression calibrator on OOF predictions...")
calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_predictions, y_train)
print(f"  ✓ Calibrator fitted on {len(y_train):,} training samples")

# Save calibrator for production use
calibrator_file = models_dir / "isotonic_calibrator.pkl"
joblib.dump(calibrator, calibrator_file)
print(f"  ✓ Saved calibrator to {calibrator_file}")

# Apply calibrator to validation set
print("\nApplying calibrator to validation set...")
y_pred_proba_calibrated = calibrator.predict(y_pred_proba_raw)
y_pred_calibrated = (y_pred_proba_calibrated >= 0.5).astype(int)

print(f"  ✓ Calibration complete")
print(f"    Raw validation range: [{y_pred_proba_raw.min():.4f}, {y_pred_proba_raw.max():.4f}]")
print(f"    Calibrated validation range: [{y_pred_proba_calibrated.min():.4f}, {y_pred_proba_calibrated.max():.4f}]")
print("=" * 60)

# Save calibrated probabilities for later use
calibrated_preds_df = pd.DataFrame({
    'WAFER_SCRIBE': val_df['WAFER_SCRIBE'],
    'y_true': y_val,
    'y_pred_proba_raw': y_pred_proba_raw,
    'y_pred_proba_calibrated': y_pred_proba_calibrated
})
calibrated_preds_df.to_parquet(output_dir / "calibrated_predictions.parquet", index=False)
print(f"\n✓ Saved calibrated predictions to {output_dir / 'calibrated_predictions.parquet'}")


Applying isotonic regression calibration (CV-based)...
Loading training data for calibration...
  Training size: 13510
  Outliers: 234 (1.73%)

Generating out-of-fold predictions using 5-fold CV...
  Fold 1/5: generating predictions for 2,702 samples...
  Fold 2/5: generating predictions for 2,702 samples...
  Fold 3/5: generating predictions for 2,702 samples...
  Fold 4/5: generating predictions for 2,702 samples...
  Fold 5/5: generating predictions for 2,702 samples...
  ✓ Out-of-fold predictions complete
    Range: [0.0115, 0.9778]

Fitting isotonic regression calibrator on OOF predictions...
  ✓ Calibrator fitted on 13,510 training samples
  ✓ Saved calibrator to outputs\models\isotonic_calibrator.pkl

Applying calibrator to validation set...
  ✓ Calibration complete
    Raw validation range: [0.0219, 0.7514]
    Calibrated validation range: [0.0000, 0.9394]

✓ Saved calibrated predictions to outputs\calibrated_predictions.parquet


In [8]:
# 1. Feature Importance Plot
print("\n1. Creating feature importance plot...")
feature_importance = model.get_feature_importance()
feature_names = feature_cols

# Create DataFrame of top 30 features
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(30)

# Color features by type
colors = []
for feat in importance_df['feature']:
    if 'LOT_ID' in feat or '__EQUIP' in feat or '__POSITION' in feat or 'lot_' in feat:
        colors.append('red')  # Lot/categorical features
    elif '__SPC' in feat:
        colors.append('green')  # SPC features
    else:
        colors.append('blue')  # Sensor features

plt.figure(figsize=(12, 10))
plt.barh(range(len(importance_df)), importance_df['importance'], color=colors)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Importance')
plt.title('Top 30 Feature Importances\n(Red=Lot/Categorical, Blue=Sensor, Green=SPC)')
plt.tight_layout()
plt.savefig(plots_dir / "feature_importance.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved feature_importance.png")


1. Creating feature importance plot...
  [OK] Saved feature_importance.png


In [9]:
# 2. Precision-Recall Curves (both raw and calibrated)
print("2. Creating precision-recall curves...")

# Raw PR curve
precision_raw, recall_raw, _ = precision_recall_curve(y_val, y_pred_proba_raw)
pr_auc_raw = auc(recall_raw, precision_raw)

# Calibrated PR curve
precision_cal, recall_cal, _ = precision_recall_curve(y_val, y_pred_proba_calibrated)
pr_auc_cal = auc(recall_cal, precision_cal)

plt.figure(figsize=(8, 6))
plt.plot(recall_raw, precision_raw, linewidth=2,
        label=f'Raw (AUC = {pr_auc_raw:.3f})', color='#2E86DE')
plt.plot(recall_cal, precision_cal, linewidth=2,
        label=f'Calibrated (AUC = {pr_auc_cal:.3f})', color='#EE5A6F', linestyle='--')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / "precision_recall_curve.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved precision_recall_curve.png")
print(f"      Raw AUC-PR = {pr_auc_raw:.3f}, Calibrated AUC-PR = {pr_auc_cal:.3f}")

2. Creating precision-recall curves...
  [OK] Saved precision_recall_curve.png
      Raw AUC-PR = 0.525, Calibrated AUC-PR = 0.612


In [10]:
# 3. ROC Curves (both raw and calibrated)
print("3. Creating ROC curves...")

# Raw ROC curve
fpr_raw, tpr_raw, _ = roc_curve(y_val, y_pred_proba_raw)
roc_auc_raw = auc(fpr_raw, tpr_raw)

# Calibrated ROC curve
fpr_cal, tpr_cal, _ = roc_curve(y_val, y_pred_proba_calibrated)
roc_auc_cal = auc(fpr_cal, tpr_cal)

plt.figure(figsize=(8, 6))
plt.plot(fpr_raw, tpr_raw, linewidth=2,
        label=f'Raw (AUC = {roc_auc_raw:.3f})', color='#2E86DE')
plt.plot(fpr_cal, tpr_cal, linewidth=2,
        label=f'Calibrated (AUC = {roc_auc_cal:.3f})', color='#EE5A6F', linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random', alpha=0.3)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(plots_dir / "roc_curve.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved roc_curve.png")
print(f"      Raw AUC-ROC = {roc_auc_raw:.3f}, Calibrated AUC-ROC = {roc_auc_cal:.3f}")

3. Creating ROC curves...
  [OK] Saved roc_curve.png
      Raw AUC-ROC = 0.887, Calibrated AUC-ROC = 0.758


## NEW: Calibration Curve

This plot shows how well-calibrated the predicted probabilities are. Points near the diagonal line indicate good calibration.

In [11]:
# 4. Calibration Curve (NEW!)
print("4. Creating calibration curve...")

# Compute calibration curves
prob_true_raw, prob_pred_raw = calibration_curve(
    y_val, y_pred_proba_raw, n_bins=10, strategy='quantile'
)
prob_true_cal, prob_pred_cal = calibration_curve(
    y_val, y_pred_proba_calibrated, n_bins=10, strategy='quantile'
)

# Calculate calibration error (mean absolute error)
cal_error_raw = np.mean(np.abs(prob_true_raw - prob_pred_raw))
cal_error_cal = np.mean(np.abs(prob_true_cal - prob_pred_cal))

plt.figure(figsize=(8, 6))
plt.plot(prob_pred_raw, prob_true_raw, 'o-', linewidth=2, markersize=8,
        label=f'Raw (MAE={cal_error_raw:.3f})', color='#2E86DE')
plt.plot(prob_pred_cal, prob_true_cal, 's-', linewidth=2, markersize=8,
        label=f'Calibrated (MAE={cal_error_cal:.3f})', color='#EE5A6F')
plt.plot([0, 1], [0, 1], '--', color='gray', linewidth=2,
        label='Perfect calibration', alpha=0.7)
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Actual Outliers')
plt.title('Calibration Curves\n(closer to diagonal = better calibrated)')
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(plots_dir / "calibration_curve.png", dpi=300, bbox_inches='tight')
plt.close()

print(f"  [OK] Saved calibration_curve.png")
print(f"      Raw calibration MAE = {cal_error_raw:.3f}")
print(f"      Calibrated calibration MAE = {cal_error_cal:.3f}")
improvement = (cal_error_raw - cal_error_cal) / cal_error_raw * 100
print(f"      Improvement: {(cal_error_raw - cal_error_cal):.3f} ({improvement:.1f}% reduction)")

4. Creating calibration curve...
  [OK] Saved calibration_curve.png
      Raw calibration MAE = 0.170
      Calibrated calibration MAE = 0.173
      Improvement: -0.003 (-1.8% reduction)


In [12]:
# 5. Confusion Matrices (both raw and calibrated)
print("5. Creating confusion matrices...")
cm_raw = confusion_matrix(y_val, y_pred_raw)
cm_cal = confusion_matrix(y_val, y_pred_calibrated)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw confusion matrix
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Non-outlier', 'Outlier'],
            yticklabels=['Non-outlier', 'Outlier'],
            ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Raw Predictions (threshold = 0.5)')

# Calibrated confusion matrix
sns.heatmap(cm_cal, annot=True, fmt='d', cmap='Oranges', cbar=False,
            xticklabels=['Non-outlier', 'Outlier'],
            yticklabels=['Non-outlier', 'Outlier'],
            ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Calibrated Predictions (threshold = 0.5)')

plt.tight_layout()
plt.savefig(plots_dir / "confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.close()
print(f"  [OK] Saved confusion_matrix.png")

5. Creating confusion matrices...
  [OK] Saved confusion_matrix.png


In [13]:
# Print classification reports
print("\n" + "=" * 60)
print("Classification Report (Raw Predictions):")
print("=" * 60)
print(classification_report(y_val, y_pred_raw, target_names=['Non-outlier', 'Outlier']))

print("\n" + "=" * 60)
print("Classification Report (Calibrated Predictions):")
print("=" * 60)
print(classification_report(y_val, y_pred_calibrated, target_names=['Non-outlier', 'Outlier']))

# Print top features
print("=" * 60)
print("Top 20 Most Important Features:")
print("=" * 60)
top_20 = importance_df.head(20)
for idx, row in top_20.iterrows():
    print(f"  {row['feature']:<60} {row['importance']:>8.2f}")


Classification Report (Raw Predictions):
              precision    recall  f1-score   support

 Non-outlier       0.99      1.00      0.99      3255
     Outlier       0.67      0.46      0.55        52

    accuracy                           0.99      3307
   macro avg       0.83      0.73      0.77      3307
weighted avg       0.99      0.99      0.99      3307


Classification Report (Calibrated Predictions):
              precision    recall  f1-score   support

 Non-outlier       0.99      1.00      0.99      3255
     Outlier       0.92      0.23      0.37        52

    accuracy                           0.99      3307
   macro avg       0.96      0.62      0.68      3307
weighted avg       0.99      0.99      0.98      3307

Top 20 Most Important Features:
  TempTr1CryoPump1st__PV0002__MEAN                                 2.70
  HeatedTCPWindowTemp__DE0004__MEAN                                2.53
  Target2DCCurrent__PV0002__MEAN                                   1.83
  Chuck

In [14]:
# Save evaluation summary
summary_file = output_dir / "evaluation_summary.txt"
print(f"\nSaving evaluation summary to {summary_file}...")

with open(summary_file, 'w') as f:
    f.write("=" * 60 + "\n")
    f.write("Model Evaluation Summary (CV-Based Calibration)\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Validation Set Size: {len(y_val)}\n")
    f.write(f"Outliers: {y_val.sum()} ({100 * y_val.mean():.2f}%)\n")
    f.write(f"Non-outliers: {(1-y_val).sum()} ({100 * (1-y_val).mean():.2f}%)\n\n")
    
    f.write("=" * 60 + "\n")
    f.write("RAW PREDICTIONS\n")
    f.write("=" * 60 + "\n")
    f.write(f"ROC AUC: {roc_auc_raw:.4f}\n")
    f.write(f"PR AUC: {pr_auc_raw:.4f}\n")
    f.write(f"Calibration MAE: {cal_error_raw:.4f}\n\n")
    
    f.write("Confusion Matrix (threshold = 0.5):\n")
    f.write(f"  True Negatives:  {cm_raw[0, 0]}\n")
    f.write(f"  False Positives: {cm_raw[0, 1]}\n")
    f.write(f"  False Negatives: {cm_raw[1, 0]}\n")
    f.write(f"  True Positives:  {cm_raw[1, 1]}\n\n")
    
    f.write("Classification Report:\n")
    f.write(classification_report(y_val, y_pred_raw, target_names=['Non-outlier', 'Outlier']))
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("CALIBRATED PREDICTIONS (CV-Based Isotonic Regression)\n")
    f.write("=" * 60 + "\n")
    f.write("NOTE: Calibrator fitted using 5-fold CV on training set\n")
    f.write("      to avoid data leakage. Applied to validation set.\n\n")
    f.write(f"ROC AUC: {roc_auc_cal:.4f}\n")
    f.write(f"PR AUC: {pr_auc_cal:.4f}\n")
    f.write(f"Calibration MAE: {cal_error_cal:.4f}\n")
    improvement_pct = 100 * (cal_error_raw - cal_error_cal) / cal_error_raw
    f.write(f"Calibration Improvement: {cal_error_raw - cal_error_cal:.4f} ({improvement_pct:.1f}% reduction)\n\n")
    
    f.write("Confusion Matrix (threshold = 0.5):\n")
    f.write(f"  True Negatives:  {cm_cal[0, 0]}\n")
    f.write(f"  False Positives: {cm_cal[0, 1]}\n")
    f.write(f"  False Negatives: {cm_cal[1, 0]}\n")
    f.write(f"  True Positives:  {cm_cal[1, 1]}\n\n")
    
    f.write("Classification Report:\n")
    f.write(classification_report(y_val, y_pred_calibrated, target_names=['Non-outlier', 'Outlier']))
    
    f.write("\n" + "=" * 60 + "\n")
    f.write("Top 20 Most Important Features:\n")
    f.write("=" * 60 + "\n")
    for idx, row in top_20.iterrows():
        f.write(f"{row['feature']:<60} {row['importance']:>8.2f}\n")

print(f"  [OK] Saved evaluation_summary.txt")


Saving evaluation summary to outputs\evaluation_summary.txt...
  [OK] Saved evaluation_summary.txt


## Append Horizon Model Summary (if available)

If horizon_results.json exists, append a summary table showing performance at each horizon.

In [15]:
# Check if horizon results exist
horizon_results_file = output_dir / "horizon_results.json"

if horizon_results_file.exists():
    print("\n" + "=" * 60)
    print("Appending horizon model summary...")
    print("=" * 60)
    
    with open(horizon_results_file, 'r') as f:
        horizon_results = json.load(f)
    
    # Get max SeqNo from column_step_map
    column_map_file = features_dir / "column_step_map.json"
    if column_map_file.exists():
        with open(column_map_file, 'r') as f:
            column_step_map = json.load(f)
        max_seqno = max(column_step_map.values())
    else:
        # Fallback: use max horizon from results
        max_seqno = max(r['horizon'] for r in horizon_results)
    
    # Create summary table
    print("\nHorizon Model Summary:")
    print("-" * 80)
    print(f"{'Horizon':<10} {'% Complete':<12} {'Features':<10} {'ROC AUC':<10} "
          f"{'PR AUC':<10} {'F1':<8}")
    print("-" * 80)
    
    for result in horizon_results:
        horizon = result['horizon']
        pct_complete = 100 * horizon / max_seqno
        n_features = result['n_features']
        roc_auc_h = result['roc_auc']
        pr_auc_h = result['pr_auc']
        f1 = result['f1']
        
        # Mark the final horizon (full model)
        marker = "  ← Full model" if horizon == max_seqno else ""
        
        print(f"{horizon:<10} {pct_complete:>6.0f}%      {n_features:<10} "
              f"{roc_auc_h:<10.3f} {pr_auc_h:<10.3f} {f1:<8.2f}{marker}")
    
    print("-" * 80)
    
    # Append to evaluation summary file
    with open(summary_file, 'a') as f:
        f.write("\n\n" + "=" * 60 + "\n")
        f.write("HORIZON MODEL SUMMARY\n")
        f.write("=" * 60 + "\n")
        f.write(f"{'Horizon':<10} | {'% Complete':<12} | {'Features':<10} | "
                f"{'ROC AUC':<10} | {'PR AUC':<10} | {'F1':<8}\n")
        f.write("-" * 80 + "\n")
        
        for result in horizon_results:
            horizon = result['horizon']
            pct_complete = 100 * horizon / max_seqno
            n_features = result['n_features']
            roc_auc_h = result['roc_auc']
            pr_auc_h = result['pr_auc']
            f1 = result['f1']
            
            marker = "  ← Full model" if horizon == max_seqno else ""
            
            f.write(f"{horizon:<10} | {pct_complete:>6.0f}%      | {n_features:<10} | "
                   f"{roc_auc_h:<10.3f} | {pr_auc_h:<10.3f} | {f1:<8.2f}{marker}\n")
        
        f.write("-" * 80 + "\n")
    
    print(f"\n[OK] Appended horizon summary to evaluation_summary.txt")
    
else:
    print("\n[INFO] No horizon_results.json found. Skipping horizon summary.")
    print("       Run step 08 (train_horizon_models) to generate horizon results.")


Appending horizon model summary...

Horizon Model Summary:
--------------------------------------------------------------------------------
Horizon    % Complete   Features   ROC AUC    PR AUC     F1      
--------------------------------------------------------------------------------
24             10%      1766       0.566      0.020      0.02    
48             20%      4373       0.630      0.023      0.03    
72             30%      6282       0.797      0.444      0.48    
96             40%      7249       0.860      0.521      0.41    
120            50%      7712       0.849      0.499      0.38    
144            60%      8445       0.855      0.530      0.55    
168            70%      8928       0.861      0.511      0.35    
192            80%      9445       0.873      0.545      0.51    
216            90%      9847       0.896      0.585      0.39    
240           100%      10302      0.887      0.526      0.55      ← Full model
--------------------------------------

UnicodeEncodeError: 'charmap' codec can't encode character '\u2190' in position 77: character maps to <undefined>

In [16]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Evaluation complete in {elapsed:.2f}s")
print("=" * 60)


[OK] Evaluation complete in 80.91s


---

## Summary: CV-Based Calibration

**Cross-Validation Calibration Method:**
- **Out-of-fold predictions** from training set used to fit calibrator
- **No data leakage** - calibrator never sees validation labels
- **More realistic** calibration performance estimates

**Isotonic regression calibration improves:**
1. **Probability reliability** - predicted probabilities match actual frequencies
2. **Threshold setting** - can confidently set intervention thresholds (e.g., "flag if >60%")
3. **Decision making** - enables risk-based decisions with quantified uncertainty

**Key metrics:**
- **Calibration MAE** - measures how far predictions are from perfect calibration (lower is better)
  - **Note:** MAE = 0.000 indicates data leakage (validation set was used for calibration)
  - Typical good values: 0.01 - 0.05 for well-calibrated models
- **ROC/PR AUC** - ranking performance (unchanged by calibration)
- **Calibration curve** - visual check: points should follow diagonal line

**When to use calibrated probabilities:**
- Setting probability thresholds for interventions
- Reporting risk scores to operators
- Cost-benefit analysis requiring accurate probability estimates
- Any decision that depends on the actual probability value (not just relative ranking)

**Warning Signs:**
- If calibrated MAE = 0.000 → data leakage!
- If calibration makes predictions worse → insufficient calibration data or distributional shift